In [1]:
import introns
import importlib
from Bio import SeqIO
from collections import defaultdict
importlib.reload(introns)

<module 'introns' from '/home/semik/projekty/ml_attempts/noncanonical_introns_master/introns.py'>

In [2]:
introny_do_smieci = [
    "EL|STRG.12279.1|scaffold_10214:22508-34308 7240 7705",
    "EL|STRG.2600.1|scaffold_1522:11335-18393 4778 5136",
    "EH|STRG.28774.2|scaffold_16461:55871-65105 930 1681",
    "EH|STRG.20430.1|scaffold_10662:1293-6188 1215 1329",
    "EH|STRG.5178.2|scaffold_2234:24437-30910 3129 5355",
    "EG|STRG.2592.1|Backbone_17881:24052-33276 6432 7500",
    "EL|STRG.17997.1|scaffold_13527:2530-10532 8331 8379",
    "EL|STRG.8943.1|scaffold_6134:1102-4802 3143 3611",
    "EL|STRG.2266.1|scaffold_1302:4984-7280 2066 2683"
 ]

path = './fastas_and_gff/'
path_a = './fastas_and_gff/Alignments/'

In [3]:
genome, genes = introns.create(path_a+'alignments_all_clean_3.fasta',
                               path_a+'alignments_all_clean_3.gff',
                               'manual')

Creating genes, sequences, introns, exons and introns' class prediction


100%|██████████| 117/117 [00:02<00:00, 40.49it/s]

[CREATE] Genes, sequences, introns, exons created, introns' classes predicted


In [4]:
introns.add_manual_annotation_from_gb(path_a+'alignments_all_clean_3_genes_only.gb', genes)

{'intron_K': 348, 'intron_NK': 325, 'intron_I': 7}

In [5]:
introns_NK, introns_K, introns_I = [], [], []
genes_K, genes_NK, genes_both = [], [], []
exons = []

for gene_name, gene_obj in genes.items():
    for intron in gene_obj.introns:
        if intron.__str__() in introny_do_smieci:
            continue
        if intron.man_annotation == 'intron_NK':
            introns_NK.append(intron)
        if intron.man_annotation == 'intron_K':
            introns_K.append(intron)
        if intron.man_annotation == 'intron_I':
            introns_I.append(intron)
    for exon in gene_obj.exons:
        exons.append(exon)
print('Intron_NK', len(introns_NK))
print('Intron_K', len(introns_K))
print('intron_I', len(introns_I))
print(len(introns_I + introns_K + introns_NK))
print('exon', len(exons))
introns_all = introns_NK + introns_K + introns_I

Intron_NK 325
Intron_K 348
intron_I 7
680
exon 1123


In [6]:
#f = open("introns_for_classifier_training.fasta", 'w')
f = open("introns_for_3-class_classifier_training.fasta", 'w')
introns_all = introns_I + introns_K + introns_NK
for i in introns_all:
    #print(i)
    if i.__str__() in introny_do_smieci:
        continue
    s = i.prev_exon.sequence[-10:] + i.sequence + i.next_exon.sequence[:10]
    f.write(f""">{i.__str__()} || {i.man_annotation} \n {s} \n""")
    for v in i.variations:
        s = v.prev_exon.sequence[-10:] + v.sequence + v.next_exon.sequence[:10]
        f.write(f""">{i.__str__()} || intron_other \n {s} \n""")
f.close()

In [7]:
"""Checking whether there is an intron with itself as its variation"""
test_intron = None
for intron in introns_all:
    intron.calculate_ML_characteristic()
    for var in intron.variations:
        if intron.__str__() == var.__str__():
            test_intron = intron
            break
    if test_intron:
        break

if not test_intron:
    print("Success!")
else:
    print(test_intron)
    print(test_intron.variations)
    


    Intron in gene: EL|STRG.12279.1|scaffold_10214:22508-34308
    scaff loc: 5805-6221
    gene loc: 5305=5721
[EL|STRG.12279.1|scaffold_10214:22508-34308 5804 6220, EL|STRG.12279.1|scaffold_10214:22508-34308 5803 6219, EL|STRG.12279.1|scaffold_10214:22508-34308 5805 6221]


In [8]:
import introns
import importlib
importlib.reload(introns)

<module 'introns' from '/home/semik/projekty/ml_attempts/noncanonical_introns_master/introns.py'>

In [15]:
bugtest_genome, bugtest_genes = introns.create(path+'bugtest_EL_STRG.5724.1_scaffold_3524.fasta',
                                               path+'bugtest_EL_STRG.5724.1_scaffold_3524.gff',
                                               'manual')

#bugtest_genome, bugtest_genes = introns.HELP_load_default_genome_genes(species="bugtest", with_reversed=False)

Creating genes, sequences, introns, exons and introns' class prediction


100%|██████████| 1/1 [00:00<00:00, 447.01it/s]

[CREATE] Genes, sequences, introns, exons created, introns' classes predicted


In [16]:
print(bugtest_genes)

{'EL|STRG.5724.1|scaffold_3542:6373-8348': EL|STRG.5724.1|scaffold_3542:6373-8348 40 2015}


In [17]:
gene = bugtest_genes['EL|STRG.5724.1|scaffold_3542:6373-8348']
for intron in gene.introns:
    print(intron)
    print(intron.variations)


    Intron in gene: EL|STRG.5724.1|scaffold_3542:6373-8348
    scaff loc: 132-562
    gene loc: 92=522
[]

    Intron in gene: EL|STRG.5724.1|scaffold_3542:6373-8348
    scaff loc: 637-884
    gene loc: 597=844
[EL|STRG.5724.1|scaffold_3542:6373-8348 636 883, EL|STRG.5724.1|scaffold_3542:6373-8348 635 882, EL|STRG.5724.1|scaffold_3542:6373-8348 634 881]

    Intron in gene: EL|STRG.5724.1|scaffold_3542:6373-8348
    scaff loc: 983-1286
    gene loc: 943=1246
[EL|STRG.5724.1|scaffold_3542:6373-8348 982 1285, EL|STRG.5724.1|scaffold_3542:6373-8348 983 1286]

    Intron in gene: EL|STRG.5724.1|scaffold_3542:6373-8348
    scaff loc: 1444-1916
    gene loc: 1404=1876
[EL|STRG.5724.1|scaffold_3542:6373-8348 1443 1915, EL|STRG.5724.1|scaffold_3542:6373-8348 1444 1916, EL|STRG.5724.1|scaffold_3542:6373-8348 1445 1917]
